# TP Integrador — Análisis de Datos (CEIA · FIUBA · 4B 2026)
## Recorridos de Ecobici — Ciudad de Buenos Aires, año 2024

**Integrantes:** Juan Palomeque · Juan Pablo Alegre · Tomás Moretti

**Problema de ML supervisado:** predecir la **duración de un viaje en Ecobici** (en minutos) en el momento en que el usuario retira la bicicleta e indica a qué estación va.
- Tipo de problema: **regresión** (target = `duracion_min`, una cantidad continua).

### Índice
1. Carga y descripción general
2. Limpieza básica (tipos, duplicados, consistencia)
3. EDA univariado (numéricas, categóricas, temporales)
4. EDA bivariado / multivariado + tests y medidas de asociación
5. Valores faltantes y su tipo (MCAR / MAR / MNAR / estructural)
6. Outliers y errores
7. Planteo del problema de ML
8. Limpieza general del dataset
9. Split train / test
10. Tratamiento de faltantes
11. Tratamiento de outliers
12. Feature engineering (creación de features)
13. Discretización
14. Codificación
15. Escalado
16. Desbalance de clases: no aplica (problema de regresión)
17. Selección de features
18. Extracción de features: PCA
19. Conclusiones

> Flujo de preparación que seguimos (clases 4 a 7): **split → tratar nulos → tratar outliers → crear features → discretizar → codificar → escalar → (balancear: solo en clasificación) → reducir dimensionalidad**. Todo lo que requiere *calcular* algo (medias, percentiles, categorías, escalas) se calcula **solo en TRAIN** y se **aplica** a TEST, para no filtrar información del test (data leakage).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer
from sklearn.feature_selection import mutual_info_regression, f_regression
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 90
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

SEED = 42          # semilla para que los resultados sean reproducibles
MUESTRA = 200_000  # tamaño de muestra para gráficos / cálculos pesados (el dataset tiene 3,5 millones de filas)

# 1. Carga y descripción general del dataset
Fuente: *Buenos Aires Data — Bicicletas públicas, recorridos realizados 2024*. Cada fila es **un viaje** (recorrido).

In [ ]:
RUTA = 'badata_ecobici_recorridos_realizados_2024.csv'
df = pd.read_csv(RUTA)

print(f'Observaciones (filas): {df.shape[0]:,}')
print(f'Variables (columnas):  {df.shape[1]}')
print(f'Memoria: {df.memory_usage(deep=True).sum()/1e9:.2f} GB')
df.head()

In [ ]:
df.info()

### Tipo de cada variable (clase 1: numéricas discretas/continuas, categóricas nominales/ordinales)
Ojo: que pandas guarde algo como número **no** significa que sea una variable numérica. Los `id` son números que funcionan como **etiquetas** (categóricas nominales).

In [ ]:
tipos = pd.DataFrame({
    'dtype_pandas': df.dtypes.astype(str),
    'valores_unicos': df.nunique(),
    'tipo_estadistico': [
        'Identificador (nominal, cardinalidad extrema)',  # id_recorrido
        'Numérica continua (segundos) -> TARGET',          # duracion_recorrido
        'Fecha/hora (numérica o categórica según uso)',    # fecha_origen
        'Categórica nominal (alta cardinalidad)',          # id_estacion_origen
        'Categórica nominal (redundante con id)',          # nombre_estacion_origen
        'Categórica nominal (redundante con id)',          # direccion_estacion_origen
        'Numérica continua (coordenada)',                  # long origen
        'Numérica continua (coordenada)',                  # lat origen
        'Fecha/hora',                                      # fecha_destino
        'Categórica nominal (alta cardinalidad)',          # id_estacion_destino
        'Categórica nominal (redundante con id)',
        'Categórica nominal (redundante con id)',
        'Numérica continua (coordenada)',
        'Numérica continua (coordenada)',
        'Identificador de usuario (nominal, cardinalidad muy alta)',
        'Categórica binaria',                              # modelo_bicicleta
        'Categórica nominal',                              # genero
    ]})
tipos

**Lectura:** el dataset tiene 3,56 millones de viajes y 17 variables. Hay pocas variables realmente "medidas" (duración, coordenadas); la mayoría son **categóricas** (estaciones, usuario, género, modelo) y **fechas**. Las estaciones tienen **alta cardinalidad** (~400 valores) y el usuario **cardinalidad muy alta** (~263 mil). Esto condiciona la codificación más adelante.

# 2. Limpieza básica (data cleaning — clase 4)
Antes de explorar a fondo corregimos lo que es claramente un problema de **calidad**: tipos de datos incorrectos, duplicados y consistencia interna.

In [ ]:
# 2.1 Tipos de dato: las fechas vienen como texto -> datetime
df['fecha_origen_recorrido']  = pd.to_datetime(df['fecha_origen_recorrido'])
df['fecha_destino_recorrido'] = pd.to_datetime(df['fecha_destino_recorrido'])

# id_usuario viene como float (por los NaN de lectura); es un identificador -> entero
df['id_usuario'] = df['id_usuario'].astype('int64')

# 2.2 Duplicados
print('Filas duplicadas exactas:', df.duplicated().sum())
print('id_recorrido repetidos:  ', df['id_recorrido'].duplicated().sum())
df = df.drop_duplicates(subset='id_recorrido', keep='first').reset_index(drop=True)
print('Filas luego de quitar duplicados:', f'{len(df):,}')

In [ ]:
# 2.3 Consistencia interna
# a) ¿La duración informada coincide con (fecha_destino - fecha_origen)?
dur_calc = (df['fecha_destino_recorrido'] - df['fecha_origen_recorrido']).dt.total_seconds()
coincide = (dur_calc == df['duracion_recorrido'])
print(f"Duración coincide con la diferencia de fechas: {coincide[dur_calc.notna()].mean():.2%} de los viajes con fecha destino")

# b) ¿Cada id de estación tiene un único nombre / coordenada?
for lado in ['origen', 'destino']:
    n = df.groupby(f'id_estacion_{lado}')[[f'nombre_estacion_{lado}', f'lat_estacion_{lado}']].nunique().max()
    print(f'Máx. nombres / latitudes distintos por id_estacion_{lado}:', n.values)

# c) Período cubierto
print('Desde', df['fecha_origen_recorrido'].min(), 'hasta', df['fecha_origen_recorrido'].max())

# d) ¿Cada usuario tiene un único género?
print('Géneros distintos por usuario:', df.groupby('id_usuario')['genero'].nunique().value_counts().to_dict())

**Conclusiones de la limpieza básica**
- Había **1 registro duplicado** (mismo `id_recorrido`) → se eliminó.
- La duración es consistente con las fechas en el 99,9% de los casos: `duracion_recorrido` **es literalmente** `fecha_destino − fecha_origen`. Esto es importante: la fecha de destino **revela la target** (leakage) y no podrá usarse como feature.
- Cada `id_estacion` tiene un único nombre, dirección y coordenada → `nombre_*` y `direccion_*` son **redundantes** con el id.
- El género es un **atributo del usuario** (cada usuario tiene siempre el mismo valor). Esto nos va a servir para entender sus faltantes.

# 3. EDA univariado (clase 2)
## 3.1 Variable numérica principal: duración del viaje
Usamos **estadística clásica** (media, desvío, asimetría, curtosis) y **estadística robusta** (mediana, IQR, MAD), que no se deja arrastrar por los valores extremos.

In [ ]:
df['duracion_min'] = df['duracion_recorrido'] / 60
d = df['duracion_min']

resumen = pd.Series({
    'media': d.mean(), 'desvío estándar': d.std(),
    'mediana (Q2)': d.median(), 'Q1': d.quantile(.25), 'Q3': d.quantile(.75),
    'IQR': d.quantile(.75) - d.quantile(.25),
    'MAD': stats.median_abs_deviation(d.sample(MUESTRA, random_state=SEED)),
    'media recortada 5%': stats.trim_mean(d, 0.05),
    'mínimo': d.min(), 'máximo': d.max(),
    'asimetría': d.skew(), 'curtosis (exceso)': d.kurt()})
resumen.to_frame('duración (min)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

axes[0].hist(d, bins=200, color='steelblue')
axes[0].set_yscale('log')
axes[0].set_title('Todos los viajes (eje Y logarítmico)')
axes[0].set_xlabel('Duración (min)')

p99 = d.quantile(.99)
axes[1].hist(d[d <= p99], bins=100, color='steelblue')
axes[1].axvline(d.median(), color='red', ls='--', label=f'mediana = {d.median():.1f}')
axes[1].axvline(d.mean(), color='orange', ls='--', label=f'media = {d.mean():.1f}')
axes[1].set_title(f'Hasta el percentil 99 ({p99:.0f} min)')
axes[1].set_xlabel('Duración (min)'); axes[1].legend()

axes[2].boxplot(d[d <= p99], orientation='horizontal')
axes[2].set_title('Boxplot (hasta p99)'); axes[2].set_xlabel('Duración (min)')
plt.tight_layout(); plt.show()

**Interpretación:**
- La duración tiene **asimetría positiva extrema** (≈156) y **curtosis enorme** (distribución *leptocúrtica*, colas muy pesadas). La media (21 min) queda por encima de la mediana (≈15 min) y el desvío (≈160 min) es mucho mayor que la media: **la media y el desvío no representan bien los datos** → preferimos mediana e IQR.
- La mayoría de los viajes dura entre **8 y 25 min** (Q1–Q3).
- Se ve una **acumulación en 0–2 min** (sospechosa, la analizamos en la sección 6) y una **cola larga** con viajes de días (máx. ≈ 30 días).

## 3.2 Variables categóricas: frecuencias, moda y entropía de Shannon

In [ ]:
def entropia_shannon(serie):
    p = serie.value_counts(normalize=True)
    H = -(p * np.log2(p)).sum()
    return H, H / np.log2(len(p))   # entropía y entropía normalizada (0 = todo en una categoría, 1 = uniforme)

filas = []
for col in ['genero', 'modelo_bicicleta', 'id_estacion_origen', 'id_estacion_destino', 'id_usuario']:
    s = df[col].fillna('SIN DATO')
    vc = s.value_counts()
    H, Hn = entropia_shannon(s)
    filas.append({'variable': col, 'cardinalidad': s.nunique(), 'moda': vc.index[0],
                  'proporción de la moda': vc.iloc[0] / len(s),
                  'rango de frecuencias': vc.iloc[0] - vc.iloc[-1],
                  'entropía': H, 'entropía normalizada': Hn})
pd.DataFrame(filas)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col in zip(axes, ['genero', 'modelo_bicicleta']):
    pct = df[col].fillna('SIN DATO').value_counts(normalize=True) * 100
    pct.plot.bar(ax=ax, color='steelblue')
    ax.bar_label(ax.containers[0], fmt='%.1f%%')
    ax.set_title(f'Viajes por {col} (%)'); ax.set_ylabel('%'); ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

**Interpretación:** 60% de los viajes son de usuarios `MALE`, 32% `FEMALE`, 7,5% `OTHER` y 0,34% sin dato. El modelo de bici `FIT` (mecánica) concentra el 73% de los viajes y `ICONIC` el 27%. Las estaciones tienen entropía normalizada alta (el uso está bastante repartido entre muchas estaciones), y los usuarios también: hay muchísimos usuarios con pocos viajes.

## 3.3 Variables temporales (fecha de origen)

In [ ]:
fo = df['fecha_origen_recorrido']
dias = {0: 'Lun', 1: 'Mar', 2: 'Mié', 3: 'Jue', 4: 'Vie', 5: 'Sáb', 6: 'Dom'}

fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
fo.dt.month.value_counts().sort_index().plot(ax=axes[0], marker='o', color='steelblue')
axes[0].set_title('Viajes por mes'); axes[0].set_xlabel('Mes'); axes[0].set_ylabel('Viajes')

(fo.dt.dayofweek.value_counts(normalize=True).sort_index().rename(index=dias) * 100).plot.bar(ax=axes[1], color='steelblue')
axes[1].set_title('Viajes por día de la semana (%)'); axes[1].set_xlabel(''); axes[1].tick_params(axis='x', rotation=0)

fo.dt.hour.value_counts().sort_index().plot(ax=axes[2], marker='o', color='firebrick')
axes[2].set_title('Viajes por hora de inicio'); axes[2].set_xlabel('Hora'); axes[2].set_xticks(range(0, 24, 2))
plt.tight_layout(); plt.show()

tabla = pd.crosstab(fo.dt.hour, fo.dt.dayofweek).rename(columns=dias)
plt.figure(figsize=(12, 6))
sns.heatmap(tabla, cmap='viridis', cbar_kws={'label': 'Viajes'})
plt.title('Viajes por hora y día de la semana'); plt.ylabel('Hora'); plt.xlabel('')
plt.show()

**Interpretación:**
- **Patrón laboral muy marcado**: los días hábiles concentran ~87% de los viajes; sábado y domingo apenas ~6,5% cada uno.
- En días hábiles hay **dos picos**: ~8 h (ida al trabajo) y ~17–18 h (vuelta), el de la tarde más fuerte. Los fines de semana el uso es bajo y se reparte en la tarde. → La Ecobici se usa principalmente como **medio de transporte al trabajo/estudio**, no como paseo.
- Por mes: menos uso en invierno (jun–ago) y un salto desde septiembre (primavera).
- Esto sugiere features: **hora**, **día de semana / fin de semana**, **hora pico**, **mes** (estacionalidad).

## 3.4 Variables espaciales: ubicación de las estaciones

In [ ]:
est = (df.groupby('id_estacion_origen')
         .agg(lat=('lat_estacion_origen', 'first'), lon=('long_estacion_origen', 'first'),
              viajes=('id_recorrido', 'size'), nombre=('nombre_estacion_origen', 'first')))

fig, axes = plt.subplots(1, 2, figsize=(17, 6.5))
sc = axes[0].scatter(est['lon'], est['lat'], s=est['viajes'] / 400, c=est['viajes'], cmap='viridis', alpha=.7)
plt.colorbar(sc, ax=axes[0], label='Viajes que salen de la estación')
axes[0].set_title('Estaciones de Ecobici (tamaño/color = viajes de origen)')
axes[0].set_xlabel('Longitud'); axes[0].set_ylabel('Latitud'); axes[0].set_aspect('equal')

est.nlargest(15, 'viajes').set_index('nombre')['viajes'].sort_values().plot.barh(ax=axes[1], color='steelblue')
axes[1].set_title('Top 15 estaciones de origen'); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()

**Interpretación:** el mapa reproduce la forma de CABA. Las estaciones con más viajes se concentran en el **centro y corredores hacia el norte** (Retiro, Constitución, Pacífico, Facultad de Derecho…), zonas de trabajo y nodos de transporte. El sur y oeste tienen menos estaciones y menos uso. Las coordenadas permiten **calcular la distancia entre estaciones**, que esperamos sea la feature más importante para predecir la duración.

# 4. EDA bivariado / multivariado
Para no saturar los gráficos usamos una **muestra aleatoria** de 200 mil viajes y recortamos al p99 para *visualizar* (no para limpiar).

In [ ]:
# distancia en línea recta entre estaciones (fórmula de haversine)
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2)**2
    return 2 * 6371 * np.arcsin(np.sqrt(a))

df['distancia_km'] = haversine_km(df['lat_estacion_origen'], df['long_estacion_origen'],
                                  df['lat_estacion_destino'], df['long_estacion_destino'])
df['mismo_origen_destino'] = (df['id_estacion_origen'] == df['id_estacion_destino']).astype(int)

m = df.sample(MUESTRA, random_state=SEED)
m = m[(m['duracion_min'] > 0) & (m['duracion_min'] <= m['duracion_min'].quantile(.99))]
m['dia'] = m['fecha_origen_recorrido'].dt.dayofweek.map(dias)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 5.5))
hb = axes[0].hexbin(m['distancia_km'], m['duracion_min'], gridsize=60, bins='log', cmap='viridis')
plt.colorbar(hb, ax=axes[0], label='log10(viajes)')
axes[0].set_xlabel('Distancia origen-destino (km)'); axes[0].set_ylabel('Duración (min)')
axes[0].set_title('Distancia vs. duración (hexbin, muestra)')

sns.boxplot(data=m, x='mismo_origen_destino', y='duracion_min', ax=axes[1], showfliers=False)
axes[1].set_xticklabels(['Origen ≠ destino', 'Origen = destino'])
axes[1].set_xlabel(''); axes[1].set_title('Duración según si vuelve a la misma estación')
plt.tight_layout(); plt.show()

**Interpretación:**
- Entre distancia y duración hay una **relación positiva** (más lejos → más tiempo), pero con mucha dispersión: para una misma distancia hay viajes rápidos y viajes muy largos (paradas intermedias, paseos). La relación **no es perfectamente lineal**.
- Abajo se ve una fila de viajes de distancia > 0 y duración casi 0: velocidades imposibles (errores, sección 6).
- Hay una franja de viajes con **distancia = 0** (vuelven a la misma estación) que se comportan distinto: o duran casi 0 min (devoluciones inmediatas) o son paseos largos. Por eso `mismo_origen_destino` es un feature útil.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 4.8))
sns.boxplot(data=m, x='dia', y='duracion_min', order=list(dias.values()), ax=axes[0], showfliers=False)
axes[0].set_title('Duración por día de la semana'); axes[0].set_xlabel('')
sns.boxplot(data=m.fillna({'genero': 'SIN DATO'}), x='genero', y='duracion_min', ax=axes[1], showfliers=False)
axes[1].set_title('Duración por género'); axes[1].set_xlabel('')
sns.violinplot(data=m, x='modelo_bicicleta', y='duracion_min', ax=axes[2], cut=0)
axes[2].set_title('Duración por modelo de bicicleta'); axes[2].set_xlabel('')
plt.tight_layout(); plt.show()

**Interpretación:** los viajes de **fin de semana son más largos** (mediana mayor y más dispersión: uso recreativo). Entre géneros hay diferencias pequeñas (`FEMALE` algo más largos). La bici `ICONIC` muestra viajes algo más largos que la `FIT`. Para saber si estas diferencias son **reales o producto del azar** usamos tests de hipótesis (clase 3).

## 4.1 Tests estadísticos y medidas de asociación (clase 3)
- **ANOVA** (numérica vs. categórica): ¿la duración media difiere según género / día? + **Eta cuadrado** (tamaño del efecto).
- **Chi-cuadrado de independencia** (categórica vs. categórica): ¿género y modelo de bici están relacionados? + **V de Cramér**.
- **Spearman** (numérica vs. numérica, relación monótona, robusta a outliers).

Con millones de filas casi cualquier diferencia da "significativa" (p-valor ≈ 0), por eso miramos también el **tamaño del efecto** (eta², V de Cramér).

In [ ]:
def eta_cuadrado(categorica, numerica):
    datos = pd.DataFrame({'c': categorica, 'y': numerica}).dropna()
    media = datos['y'].mean()
    ss_entre = datos.groupby('c')['y'].apply(lambda g: len(g) * (g.mean() - media)**2).sum()
    ss_total = ((datos['y'] - media)**2).sum()
    return ss_entre / ss_total

def cramers_v(x, y):
    tabla = pd.crosstab(x, y)
    chi2_, p, _, esperadas = stats.chi2_contingency(tabla)
    n = tabla.values.sum(); k = min(tabla.shape) - 1
    return np.sqrt(chi2_ / (n * k)), chi2_, p, (esperadas >= 5).mean()

# ANOVA + eta²
for col in ['genero', 'dia', 'modelo_bicicleta']:
    grupos = [g['duracion_min'].values for _, g in m.dropna(subset=[col]).groupby(col)]
    F, p = stats.f_oneway(*grupos)
    print(f'ANOVA duración ~ {col:17s}: F = {F:9.1f}  p-valor = {p:.2e}  eta² = {eta_cuadrado(m[col], m["duracion_min"]):.4f}')

# Chi-cuadrado + V de Cramér
v, c2, p, cochran = cramers_v(df['genero'].dropna(), df.loc[df['genero'].notna(), 'modelo_bicicleta'])
print(f'\nChi² genero x modelo: chi² = {c2:,.0f}  p-valor = {p:.2e}  V de Cramér = {v:.3f}  (frec. esperadas >= 5: {cochran:.0%})')

# Spearman distancia - duración
rho, p = stats.spearmanr(m['distancia_km'], m['duracion_min'])
print(f'\nSpearman distancia ~ duración: rho = {rho:.3f}  p-valor = {p:.2e}')

**Interpretación:**
- Los ANOVA rechazan H₀ (p < 0,05): la duración media **sí difiere** entre géneros, días y modelos. Pero los **eta² son chicos**: género (≈0,005) y modelo (≈0,0003) explican menos del 1% de la variabilidad de la duración (efecto despreciable); el día de la semana explica ≈4,5% (efecto pequeño, el mayor de los tres).
- Chi²: género y modelo de bici no son independientes, pero la **V de Cramér ≈ 0** → asociación despreciable (no hay redundancia entre ellas). Se cumple la regla de Cochran.
- Spearman distancia–duración ≈ **0,59** (relación monótona positiva, moderada-fuerte) → la distancia es la variable con más relación con la target.
- *Nota sobre supuestos:* ANOVA asume normalidad y varianzas homogéneas; la duración es muy asimétrica, así que el resultado se toma como orientativo (con n tan grande el test es robusto, pero lo importante es el tamaño del efecto).

# 5. Valores faltantes y su tipo (clase 3)
Recordatorio: **MCAR** (falta al azar, independiente de todo) · **MAR** (la falta se relaciona con otras columnas observadas) · **MNAR** (falta por el propio valor no observado) · **Estructural** (el dato no existe).

In [ ]:
faltantes = pd.DataFrame({'faltantes': df.isna().sum(), '%': df.isna().mean() * 100})
faltantes[faltantes['faltantes'] > 0]

También hay **faltantes "disfrazados"**: viajes con `duracion_recorrido = 0` (la duración no se registró realmente). Los analizamos junto con `fecha_destino`.

## 5.1 `fecha_destino_recorrido` (3.379 nulos, 0,09%)
Para saber si es MCAR, agrupamos de distintas maneras y vemos si el % de faltantes es parecido en todos los grupos (criterio de clase).

In [ ]:
falta_fd = df['fecha_destino_recorrido'].isna()
mes = df['fecha_origen_recorrido'].dt.month

fig, axes = plt.subplots(1, 2, figsize=(17, 4.5))
(falta_fd.groupby(mes).mean() * 100).plot.bar(ax=axes[0], color='firebrick')
axes[0].set_title('% de viajes sin fecha de destino, por mes'); axes[0].set_xlabel('Mes'); axes[0].set_ylabel('%')
axes[0].tick_params(axis='x', rotation=0)

por_dia = falta_fd.groupby(df['fecha_origen_recorrido'].dt.date).sum()
por_dia.plot(ax=axes[1], color='firebrick')
axes[1].set_title('Cantidad de viajes sin fecha de destino, por día'); axes[1].set_xlabel('')
plt.tight_layout(); plt.show()

print('Primer viaje sin fecha de destino:', df.loc[falta_fd, 'fecha_origen_recorrido'].min())
print('Duración de esos viajes:', df.loc[falta_fd, 'duracion_recorrido'].describe()[['min', 'max']].to_dict())

# Test chi² de independencia: ¿la falta depende del mes?
v, c2, p, _ = cramers_v(falta_fd, mes)
print(f'Chi² faltante_fecha_destino x mes: p-valor = {p:.2e}, V de Cramér = {v:.3f}')

**Diagnóstico → MAR (no MCAR).**
- **Todos** los faltantes de `fecha_destino` aparecen **a partir del 7/11/2024** y solo en noviembre–diciembre; de enero a octubre hay 0. El % de faltantes **no** es parecido entre meses y el chi² rechaza la independencia → la falta **depende de una variable observada (la fecha)**: es **MAR**. La explicación más probable es un **cambio o falla del sistema de registro** a partir de esa fecha.
- Además, **todos** esos viajes tienen `duracion = 0`: el viaje no quedó "cerrado" en el sistema. No es que no sepamos la fecha: **no sabemos la duración, que es nuestra target**.
- *Corrección respecto del borrador:* lo habíamos llamado MCAR, pero el patrón temporal muestra que no lo es.

**Tratamiento:** se **eliminan** estas filas. Justificación: (1) son el 0,09% del total; (2) **no tienen target** (duración 0 no es un valor real) y un registro sin target no sirve para entrenar ni evaluar un modelo supervisado; (3) imputar la target sería inventar la respuesta.

## 5.2 `genero` (11.946 nulos, 0,34%)

In [ ]:
falta_g = df['genero'].isna()
agrupaciones = {
    'mes': mes,
    'día de semana': df['fecha_origen_recorrido'].dt.dayofweek.map(dias),
    'modelo_bicicleta': df['modelo_bicicleta'],
    'hora (franja de 6 h)': (df['fecha_origen_recorrido'].dt.hour // 6) * 6,
}
fig, axes = plt.subplots(1, 4, figsize=(20, 3.8), sharey=True)
for ax, (nombre, g) in zip(axes, agrupaciones.items()):
    (falta_g.groupby(g).mean() * 100).plot.bar(ax=ax, color='slategray')
    ax.axhline(falta_g.mean() * 100, color='red', ls='--')
    ax.set_title(f'% sin género por {nombre}'); ax.set_xlabel(''); ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

for nombre, g in agrupaciones.items():
    v, _, p, _ = cramers_v(falta_g, g)
    print(f'falta de género x {nombre:20s} V de Cramér = {v:.4f}  (p-valor = {p:.2e})')

usuarios_sin_genero = df.loc[falta_g, 'id_usuario'].unique()
print(f'\nUsuarios distintos con género faltante: {len(usuarios_sin_genero)}')
print('¿Alguno de esos usuarios tiene género informado en otro viaje?',
      df.loc[df['id_usuario'].isin(usuarios_sin_genero), 'genero'].notna().any())

**Diagnóstico → MCAR respecto de las variables del viaje (aprox.), con origen en el perfil del usuario.**
- El % de faltantes es **prácticamente igual** en todos los meses, días, modelos y franjas horarias (≈0,3%) y las V de Cramér son ≈ 0: la falta **no depende** de las características del viaje → se comporta como **MCAR**. (Los p-valores pueden dar "significativos" por el enorme n, pero el tamaño del efecto es despreciable.)
- Los faltantes corresponden a **535 usuarios que nunca informaron su género** (ninguno lo tiene en otro viaje). Es un dato que falta **a nivel usuario** (perfil incompleto), no un error al azar de cada viaje. No hay evidencia de MNAR.

**Tratamiento (sección 10):** como es una categórica con muy baja cantidad de faltantes (<10%), según la tabla de la clase 5 podríamos eliminar filas o imputar con la moda / nueva categoría. Elegimos **crear la categoría `NO_INFORMADO`**: no perdemos los viajes de esos usuarios y no inventamos un género (imputar `MALE` por moda sería asignarle un valor falso al 100% de los viajes de 535 personas).

# 6. Outliers y errores
Distinguimos dos cosas (clase 4: *"analizar si se trata de errores o son valores legítimos"*):
1. **Errores / registros inválidos** → se eliminan en la limpieza.
2. **Valores extremos legítimos** → se tratan (capping) después del split.

## 6.1 Detección estadística: IQR (robusto) y media ± 3σ (clásico)

In [ ]:
def limites_iqr(s):
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

filas = []
for col in ['duracion_min', 'distancia_km', 'lat_estacion_origen', 'long_estacion_origen']:
    s = df[col]
    li, ls_ = limites_iqr(s)
    mu, sd = s.mean(), s.std()
    filas.append({'variable': col,
                  'lim_inf_IQR': li, 'lim_sup_IQR': ls_, 'outliers_IQR': ((s < li) | (s > ls_)).sum(),
                  'lim_sup_3σ': mu + 3 * sd, 'outliers_3σ': ((s < mu - 3 * sd) | (s > mu + 3 * sd)).sum()})
pd.DataFrame(filas)

- **Duración:** el método IQR marca como outliers los viajes de más de ~49 min (≈198 mil viajes, 5,6%). El método de 3σ da un límite absurdo (≈ 8 horas) porque la media y el desvío están inflados por los extremos y además **asume normalidad**, que acá no se cumple → **el IQR es más adecuado**.
- **Latitud / distancia:** los "outliers" son estaciones reales del extremo sur de la ciudad o viajes largos legítimos (se ven en el mapa de la sección 3.4) → **no son errores, se conservan**.

## 6.2 Análisis de errores en la duración (con conocimiento del dominio)

In [ ]:
d = df['duracion_min']
mismo = df['mismo_origen_destino'] == 1
# Tramos de duración (en minutos). Los viajes de más de 1 hora se abren en subcategorías:
# 1-3 h, 3-6 h, 6-12 h, 12-24 h y más de 24 h
CORTES_TRAMOS = [-0.01, 0, 1, 2, 5, 10, 60, 3 * 60, 6 * 60, 12 * 60, 24 * 60, np.inf]
ETIQ_TRAMOS = ['0', '(0,1] min', '(1,2] min', '(2,5] min', '(5,10] min', '(10,60] min',
               '(1,3] h', '(3,6] h', '(6,12] h', '(12,24] h', '> 24 h']
tramos = pd.cut(d, CORTES_TRAMOS, labels=ETIQ_TRAMOS)
tabla_tramos = pd.crosstab(tramos, mismo.map({True: 'Origen = destino', False: 'Origen ≠ destino'}), margins=True)
tabla_tramos['% del total'] = tabla_tramos['All'] / len(df) * 100
tabla_tramos

### 6.2.1 Viajes largos: ¿paseos reales o errores?
Para los viajes de más de 1 hora no alcanza con la duración: miramos **cuándo empiezan, cuándo terminan, si cruzan la medianoche, si son de fin de semana y si vuelven a la misma estación**. Si fueran paseos, deberían parecerse a un uso recreativo de día. Si la bici quedó "retenida" (el usuario se la llevó a su casa, no la ancló bien, la dejó estacionada), deberían empezar a la tarde-noche y terminar a la mañana siguiente.

In [ ]:
largos = df[d > 60].copy()
largos['tramo'] = pd.cut(largos['duracion_min'], [60, 3 * 60, 6 * 60, 12 * 60, 24 * 60, np.inf],
                         labels=['(1,3] h', '(3,6] h', '(6,12] h', '(12,24] h', '> 24 h'])
largos['hora_inicio'] = largos['fecha_origen_recorrido'].dt.hour
largos['hora_fin'] = largos['fecha_destino_recorrido'].dt.hour
largos['cruza_medianoche'] = largos['fecha_origen_recorrido'].dt.date != largos['fecha_destino_recorrido'].dt.date
largos['fin_de_semana'] = largos['fecha_origen_recorrido'].dt.dayofweek >= 5

g = largos.groupby('tramo', observed=True)
perfil = pd.DataFrame({
    'viajes': g.size(),
    '% del total de viajes': g.size() / len(df) * 100,
    '% cruza la medianoche': g['cruza_medianoche'].mean() * 100,
    '% fin de semana': g['fin_de_semana'].mean() * 100,
    '% origen = destino': g['mismo_origen_destino'].mean() * 100,
    'hora de inicio (mediana)': g['hora_inicio'].median(),
    'hora de fin (mediana)': g['hora_fin'].median(),
})
display(perfil.round(2))

fig, axes = plt.subplots(1, 3, figsize=(20, 4.5))
for tramo, color in zip(['(1,3] h', '(3,6] h', '(6,12] h', '(12,24] h'], ['#4C72B0', '#55A868', '#DD8452', '#C44E52']):
    sub = largos[largos['tramo'] == tramo]
    axes[0].plot(sub['hora_inicio'].value_counts(normalize=True).sort_index() * 100, marker='o', label=tramo, color=color)
    axes[1].plot(sub['hora_fin'].value_counts(normalize=True).sort_index() * 100, marker='o', label=tramo, color=color)
for ax, t in zip(axes[:2], ['Hora de INICIO del viaje', 'Hora de FIN del viaje']):
    ax.set_title(t); ax.set_xlabel('Hora'); ax.set_ylabel('% de los viajes del tramo'); ax.set_xticks(range(0, 24, 2)); ax.legend()
perfil['% cruza la medianoche'].plot.bar(ax=axes[2], color=['#4C72B0', '#55A868', '#DD8452', '#C44E52', '#8172B3'])
axes[2].set_title('% de viajes que terminan otro día'); axes[2].set_ylabel('%'); axes[2].set_xlabel(''); axes[2].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

# ¿Hay un "corte natural" en la distribución? Viajes por media hora entre 1 y 24 h
plt.figure(figsize=(14, 3.5))
plt.hist(largos.loc[largos['duracion_min'] <= 24 * 60, 'duracion_min'] / 60, bins=np.arange(1, 24.5, 0.5), color='steelblue')
plt.yscale('log'); plt.axvline(6, color='red', ls='--', label='6 h')
plt.title('Viajes de 1 a 24 horas (eje Y logarítmico)'); plt.xlabel('Duración (horas)'); plt.ylabel('viajes'); plt.legend()
plt.show()

In [ ]:
# velocidad implícita: distancia en línea recta / tiempo (solo para detectar errores, NO es un feature: usa la target)
vel = df['distancia_km'] / (df['duracion_recorrido'] / 3600)
print('Viajes con velocidad implícita > 25 km/h:', ((vel > 25) & np.isfinite(vel)).sum())

fig, ax = plt.subplots(figsize=(10, 4))
corta = df[(d > 0) & (d <= 10)]
sns.histplot(data=corta, x='duracion_min', hue=corta['mismo_origen_destino'].map({1: 'Origen = destino', 0: 'Origen ≠ destino'}),
             bins=100, ax=ax)
ax.set_title('Viajes de 0 a 10 minutos'); ax.set_xlabel('Duración (min)')
plt.show()

**Hallazgos (registros que consideramos errores o "no viajes"):**
| Caso | Cantidad aprox. | Interpretación | Decisión |
|---|---|---|---|
| Duración = 0 | 37,6 mil | No se registró la duración (incluye los 3.379 sin fecha de destino) | Eliminar |
| Origen = destino y duración ≤ 2 min | ~250 mil | El usuario saca la bici y la devuelve enseguida (bici con falla, arrepentimiento). No es un viaje | Eliminar |
| Velocidad implícita > 25 km/h | decenas de miles | Físicamente imposible en bici urbana (ej.: 2 km en 30 s) → error de registro | Eliminar |
| (1, 3] h | ~105 mil (3%) | Uso recreativo: 41% de fin de semana, 24% vuelve a la misma estación, empieza a la tarde y termina el mismo día → **paseos reales** | Mantener (capping en sección 11) |
| (3, 6] h | ~7.900 (0,2%) | Todavía parecido a un paseo largo con paradas: empieza a media tarde (mediana 14 h), termina a la tarde (mediana 17 h), solo 13% cruza la medianoche → **extremo pero posible** | Mantener (capping en sección 11) |
| (6, 12] h | ~2.400 (0,07%) | Cambia el patrón: empieza a la tarde-noche (las horas más frecuentes son 18–21 h) y termina de madrugada o a la mañana (mediana 9 h); 47% cruza la medianoche → la bici quedó **retenida** (en casa, mal anclada) | Eliminar |
| (12, 24] h | ~860 (0,02%) | Empieza a la tarde-noche y se devuelve a la mañana siguiente; 86% cruza la medianoche → **retención nocturna** | Eliminar |
| > 24 h | ~690 (0,02%) | Bici no devuelta durante días / falla del sistema | Eliminar |

El pico de viajes de < 2 min es casi todo de origen = destino, lo que confirma la interpretación de "devolución inmediata".

**Sobre los viajes largos, una distinción importante:** la duración registrada no es *tiempo pedaleando*, es *tiempo que la bici estuvo fuera del anclaje*. Por eso un viaje de 8 horas no es necesariamente un error **de registro**: muy probablemente las fechas son correctas y la persona se llevó la bici a su casa. Lo que sí es claro es que **no representa un viaje en bicicleta**, que es lo que queremos predecir. Lo eliminamos por estar **fuera del alcance del problema**, no por ser un dato falso.

**¿Por qué el corte en 6 horas?** La distribución decrece en forma continua, sin un salto natural: cualquier umbral es una decisión de criterio, y hay que justificarla. Elegimos 6 h porque es donde **cambia el comportamiento**: hasta 6 h los viajes terminan a la tarde del mismo día (patrón de paseo); desde 6 h terminan de madrugada o a la mañana siguiente (el % que cruza la medianoche salta de 13% a 47% y luego a 86%). La regla de dominio ("nadie pedalea más de 6 horas seguidas") coincide con lo que muestran los datos.

# 7. Planteo del problema de ML supervisado

**Escenario:** un usuario retira una bicicleta e indica a qué estación se dirige. Queremos **predecir cuánto va a durar el viaje**. Utilidad: estimar cuándo se liberará una bici / anclaje en la estación de destino, planificar la redistribución de bicicletas, avisar al usuario el tiempo estimado.

- **Tipo de problema:** **regresión** (la etiqueta es una cantidad).
- **Variable target:** `duracion_min` = duración del viaje en minutos.
- **¿Por qué regresión y no clasificación?** La duración es continua y el escenario pide estimar minutos. Convertirla en clases (ej.: corto/medio/largo) perdería información y obligaría a elegir cortes arbitrarios.

**Features candidatas** (información disponible **al momento de retirar la bici**):
- Espaciales: estación de origen y de destino, sus coordenadas, **distancia** entre ellas, si vuelve a la misma estación.
- Temporales: hora, día de la semana, mes, fin de semana, feriado, hora pico.
- Usuario y bici: género, modelo de bicicleta, frecuencia de uso del usuario.

**Excluidas:**
| Columna | Motivo |
|---|---|
| `fecha_destino_recorrido` | **Leakage**: duración = destino − origen. No se conoce al retirar la bici |
| `duracion_recorrido` | Es la target (en segundos) |
| `id_recorrido` | Identificador sin información (cardinalidad extrema) |
| `nombre_*`, `direccion_*` | Redundantes con `id_estacion_*` (relación 1 a 1) |
| `id_usuario` (crudo) | Cardinalidad muy alta; se usa solo para crear un feature agregado |

# 8. Limpieza general del dataset
Esta limpieza se hace **antes del split** porque son reglas **fijas, fila por fila** (no se calcula nada a partir de otras observaciones), por lo que no generan leakage: eliminamos registros que no son viajes válidos en cualquier conjunto.

In [ ]:
n0 = len(df)
vel = df['distancia_km'] / (df['duracion_recorrido'] / 3600)

reglas = {
    'Sin fecha de destino (target faltante)': df['fecha_destino_recorrido'].isna(),
    'Duración = 0': df['duracion_recorrido'] == 0,
    'Devolución inmediata (origen = destino y ≤ 2 min)': (df['mismo_origen_destino'] == 1) & (df['duracion_min'] <= 2),
    'Velocidad imposible (> 25 km/h)': (vel > 25) & np.isfinite(vel),
    'Duración > 6 h (bici retenida o no devuelta, ver 6.2.1)': df['duracion_min'] > 6 * 60,
}
a_borrar = pd.Series(False, index=df.index)
embudo = []
for nombre, regla in reglas.items():
    nuevas = regla & ~a_borrar
    embudo.append({'regla': nombre, 'filas eliminadas (nuevas)': nuevas.sum()})
    a_borrar |= regla
embudo = pd.DataFrame(embudo)
embudo.loc[len(embudo)] = ['TOTAL', a_borrar.sum()]
embudo['% del total'] = embudo['filas eliminadas (nuevas)'] / n0 * 100
display(embudo)

df = df[~a_borrar].copy()

# Eliminamos columnas irrelevantes / redundantes / con leakage
cols_drop = ['id_recorrido', 'duracion_recorrido', 'fecha_destino_recorrido',
             'nombre_estacion_origen', 'direccion_estacion_origen',
             'nombre_estacion_destino', 'direccion_estacion_destino']
df = df.drop(columns=cols_drop).reset_index(drop=True)
print(f'Filas: {n0:,} -> {len(df):,}  ({len(df)/n0:.1%} conservado)')
print('Columnas restantes:', df.columns.tolist())

**Resultado:** se eliminaron 354.831 registros (10,0%) y se conservan 3.204.452 viajes (90,0%). Lo eliminado son registros que no representan un viaje real o que no tienen target. Con más de 3 millones de filas restantes, la pérdida no compromete el análisis.

In [ ]:
# Distribución de la target después de la limpieza
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].hist(df['duracion_min'], bins=150, color='steelblue'); axes[0].set_yscale('log')
axes[0].set_title('Duración luego de la limpieza (eje Y log)'); axes[0].set_xlabel('min')
axes[1].hist(np.log1p(df['duracion_min']), bins=100, color='seagreen')
axes[1].set_title('log(1 + duración)'); axes[1].set_xlabel('log(min)')
plt.tight_layout(); plt.show()
print(f"Asimetría: original = {df['duracion_min'].skew():.2f} | log = {np.log1p(df['duracion_min']).skew():.2f}")

La transformación **logarítmica** vuelve la distribución mucho más simétrica. Queda como opción para el modelo (predecir `log(duración)`), útil para modelos lineales que son sensibles a la asimetría y a los extremos.

## 8.1 Cruce: día de la semana × duración del viaje
Con los datos ya limpios cruzamos el día de la semana con la duración. Usamos **mediana y cuartiles** (la duración es asimétrica), el **% de viajes largos** y la **distancia mediana**, para ver si los viajes del fin de semana son más largos porque se va más lejos o porque se usa la bici de otra forma.

In [ ]:
dias_lista = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
dow = df['fecha_origen_recorrido'].dt.dayofweek
g = df.groupby(dow)
cruce = pd.DataFrame({
    'viajes': g.size(),
    '% de los viajes': g.size() / len(df) * 100,
    'Q1 (min)': g['duracion_min'].quantile(.25),
    'mediana (min)': g['duracion_min'].median(),
    'Q3 (min)': g['duracion_min'].quantile(.75),
    'media (min)': g['duracion_min'].mean(),
    '% > 30 min': g['duracion_min'].apply(lambda s: (s > 30).mean() * 100),
    '% > 60 min': g['duracion_min'].apply(lambda s: (s > 60).mean() * 100),
    'distancia mediana (km)': g['distancia_km'].median(),
})
cruce.index = dias_lista
display(cruce.round(1))

print(f"eta² (día de la semana -> duración): {eta_cuadrado(dow, df['duracion_min']):.3f}")
finde = dow >= 5
print(f"Mediana días hábiles: {df.loc[~finde, 'duracion_min'].median():.1f} min | fin de semana: {df.loc[finde, 'duracion_min'].median():.1f} min")

mapa = df.pivot_table(index=df['fecha_origen_recorrido'].dt.hour, columns=dow, values='duracion_min', aggfunc='median')
mapa.columns = dias_lista

fig, axes = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [1, 1.15]})
sns.boxplot(x=dow.map(dict(enumerate(dias_lista))), y=df['duracion_min'], order=dias_lista,
            showfliers=False, ax=axes[0], color='#A9C4E2', width=.6)
for i, v in enumerate(cruce['mediana (min)']):
    axes[0].text(i, cruce['Q3 (min)'].iloc[i] + 1, f'mediana {v:.1f}', ha='center', fontsize=9)
axes[0].set_title('Duración por día de la semana (outliers no dibujados)')
axes[0].set_xlabel(''); axes[0].set_ylabel('Duración (min)')
sns.heatmap(mapa, cmap='Blues', ax=axes[1], cbar_kws={'label': 'Duración mediana (min)'})
axes[1].set_title('Duración mediana según hora de inicio y día'); axes[1].set_xlabel(''); axes[1].set_ylabel('Hora de inicio')
plt.tight_layout(); plt.show()

**Interpretación:**
- **De lunes a viernes la duración es casi idéntica**: mediana ≈ 15 min, Q1–Q3 ≈ 10–24 min y ~15% de viajes de más de 30 min.
- **El fin de semana los viajes son mucho más largos**: mediana de 24 min el sábado y 26,5 min el domingo (+60–70%), y el % de viajes de más de 30 min casi se triplica (38–43%).
- La **distancia mediana casi no cambia** (1,7 km en días hábiles vs. 1,9 km el fin de semana). Los viajes no son más largos porque se va más lejos, sino porque **se va más lento o con paradas**: uso recreativo (paseo) en vez de traslado.
- El mapa de calor lo confirma: en días hábiles los viajes de la mañana (6–9 h, ida al trabajo) son los más cortos (~12–13 min). Sábado y domingo entre las 10 y las 18 h superan los 25–30 min.
- El día explica ≈ 4,7% de la variabilidad de la duración (eta²): es un efecto **pequeño pero el mayor entre las variables temporales**. Por eso conviene `es_fin_de_semana` como feature, y además su **interacción con la hora** (un modelo de árboles la captura sola).

# 9. Split del dataset (train / test)
- Proporción **80 / 20** (clase 4).
- **Split aleatorio estratificado por tramos de duración** (≤15, 15–30, 30–60 y >60 min). En regresión no hay clases, pero estratificar por tramos de la target es una práctica habitual: garantiza que la distribución de la duración —incluida su cola de viajes largos— quede igual en train y test. Los tramos se usan **solo para el split**, no son una feature ni una target.
- ¿Por qué no un split temporal? El objetivo no es pronosticar el futuro de una serie, sino estimar la duración de un viaje a partir de sus características; queremos que train y test tengan todas las estaciones del año. (Un split temporal —ej.: entrenar ene–oct y testear nov–dic— sería una alternativa válida para simular uso en producción.)

**A partir de acá, todo lo que se calcula se calcula en TRAIN y se aplica en TEST.**

In [ ]:
CORTES = [0, 15, 30, 60, np.inf]
TRAMOS = ['≤15 min', '15-30 min', '30-60 min', '>60 min']
estrato = pd.cut(df['duracion_min'], bins=CORTES, labels=TRAMOS)   # solo para estratificar el split

train, test = train_test_split(df, test_size=0.2, random_state=SEED, stratify=estrato)
train, test = train.copy(), test.copy()
del df  # liberamos memoria

print(f'TRAIN: {len(train):,} filas | TEST: {len(test):,} filas')
display(pd.DataFrame({'train %': pd.cut(train['duracion_min'], CORTES, labels=TRAMOS).value_counts(normalize=True) * 100,
                      'test %':  pd.cut(test['duracion_min'],  CORTES, labels=TRAMOS).value_counts(normalize=True) * 100}).round(2))
pd.DataFrame({'train': train['duracion_min'].describe(), 'test': test['duracion_min'].describe()}).round(2)

# 10. Tratamiento de valores faltantes
Luego de la limpieza, el único faltante es `genero`. Aplicamos la decisión de la sección 5.2: nueva categoría `NO_INFORMADO` (misma regla en train y test).

In [ ]:
print('Nulos en train antes:\n', train.isna().sum()[train.isna().sum() > 0])
for d_ in (train, test):
    d_['genero'] = d_['genero'].fillna('NO_INFORMADO')
print('\nNulos totales después -> train:', train.isna().sum().sum(), '| test:', test.isna().sum().sum())
train['genero'].value_counts(normalize=True).mul(100).round(2)

# 11. Tratamiento de outliers
Los errores ya se eliminaron en la limpieza. Quedan **valores extremos legítimos** en la target (viajes de varias horas). Opciones vistas en clase: eliminar, imputar, **capping/winsorización**, transformación logarítmica, segmentación.

**Elegimos capping en el percentil 99 calculado en TRAIN.** Justificación:
- No son errores, así que no los eliminamos (perderíamos información de que "ese viaje fue muy largo").
- Pero unos pocos viajes de 10–20 horas dominarían una función de pérdida como el error cuadrático y distorsionarían el modelo.
- El capping conserva la fila y su orden ("fue de los más largos") y limita su peso.
- El umbral (p99) se calcula **solo en train** y se aplica igual en test.

Sobre las features: la `distancia_km` tiene outliers por IQR pero son viajes reales y acotados (máx. ~20 km), por lo que no se tratan.

In [ ]:
umbral_p99 = train['duracion_min'].quantile(0.99)   # CALCULAR en train
print(f'Umbral de capping (p99 de train): {umbral_p99:.1f} min')
print(f"Viajes afectados -> train: {(train['duracion_min'] > umbral_p99).sum():,} | test: {(test['duracion_min'] > umbral_p99).sum():,}")

fig, axes = plt.subplots(1, 2, figsize=(15, 3.5))
axes[0].boxplot(train['duracion_min'], orientation='horizontal'); axes[0].set_title('Antes del capping (train)')
for d_ in (train, test):                                   # APLICAR en train y test
    d_['duracion_min'] = d_['duracion_min'].clip(upper=umbral_p99)
axes[1].boxplot(train['duracion_min'], orientation='horizontal'); axes[1].set_title('Después del capping (train)')
for ax in axes: ax.set_xlabel('Duración (min)')
plt.tight_layout(); plt.show()

# 12. Feature engineering: creación de nuevos features (clase 5)
| Feature | Técnica | Justificación |
|---|---|---|
| `distancia_km` | Matemática (haversine con lat/lon) | El tiempo de viaje depende sobre todo de la distancia. Resume 4 coordenadas en 1 valor con significado (como el IMC en clase) |
| `mismo_origen_destino` | Comparación A = B | Los viajes que vuelven a la misma estación son paseos: la distancia es 0 pero la duración no |
| `hora`, `dia_semana`, `mes` | Temporal (extraer de timestamp) | Vimos patrones fuertes por hora, día y estación del año |
| `es_fin_de_semana`, `es_feriado` | Temporal + dominio | Uso recreativo → viajes más largos |
| `es_hora_pico` | Temporal + dominio | Lun–vie 7–9 h y 17–19 h: viajes al trabajo, más directos |
| `viajes_usuario` | Agregado (conteo por usuario), **calculado en train** | Un usuario frecuente conoce el recorrido y usa la bici como transporte |
| `te_estacion_origen`, `te_estacion_destino` | Target encoding (media de duración por estación), **calculado en train** | Ver sección 14 |

Las features que **no dependen de otras observaciones** (distancia, hora…) se calculan igual en ambos conjuntos. Las que **sí dependen** (conteo por usuario) se calculan en train y se *mapean* a test (usuarios nuevos en test → 0 viajes previos conocidos).

In [ ]:
# Feriados nacionales de Argentina 2024 (incluye días no laborables con fines turísticos)
# Fuente: calendario oficial de feriados 2024 — verificar contra argentina.gob.ar
FERIADOS_2024 = pd.to_datetime([
    '2024-01-01', '2024-02-12', '2024-02-13', '2024-03-24', '2024-03-28', '2024-03-29',
    '2024-04-01', '2024-04-02', '2024-05-01', '2024-05-25', '2024-06-17', '2024-06-20',
    '2024-06-21', '2024-07-09', '2024-08-17', '2024-10-11', '2024-10-12', '2024-11-18',
    '2024-12-08', '2024-12-25']).date

def features_temporales(d_):
    f = d_['fecha_origen_recorrido']
    d_['hora'] = f.dt.hour
    d_['dia_semana'] = f.dt.dayofweek          # 0 = lunes
    d_['mes'] = f.dt.month
    d_['es_fin_de_semana'] = (d_['dia_semana'] >= 5).astype(int)
    d_['es_feriado'] = f.dt.date.isin(FERIADOS_2024).astype(int)
    d_['es_hora_pico'] = ((d_['dia_semana'] < 5) & (d_['es_feriado'] == 0) &
                          d_['hora'].isin([7, 8, 17, 18])).astype(int)
    return d_

train = features_temporales(train)
test = features_temporales(test)

# Agregado por usuario: CALCULAR en train, APLICAR en test
viajes_por_usuario = train['id_usuario'].value_counts()
train['viajes_usuario'] = train['id_usuario'].map(viajes_por_usuario)
test['viajes_usuario'] = test['id_usuario'].map(viajes_por_usuario).fillna(0)

train[['distancia_km', 'mismo_origen_destino', 'hora', 'dia_semana', 'mes', 'es_fin_de_semana',
       'es_feriado', 'es_hora_pico', 'viajes_usuario']].describe().T

In [ ]:
# ¿Los nuevos features se relacionan con la target? (muestra de train)
mt = train.sample(MUESTRA, random_state=SEED)
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, col in zip(axes, ['es_fin_de_semana', 'es_feriado', 'es_hora_pico', 'mismo_origen_destino']):
    sns.boxplot(data=mt, x=col, y='duracion_min', ax=ax, showfliers=False)
    ax.set_title(f'Duración según {col}')
plt.tight_layout(); plt.show()

mt['viajes_usuario_q'] = pd.qcut(mt['viajes_usuario'], 5, duplicates='drop')
mt.groupby('viajes_usuario_q', observed=True)['duracion_min'].median().plot.bar(figsize=(8, 3.5), color='steelblue')
plt.title('Duración mediana según frecuencia de uso del usuario (quintiles)'); plt.xlabel('viajes del usuario en train')
plt.xticks(rotation=20); plt.show()

**Interpretación:** fines de semana, feriados y viajes que vuelven al mismo lugar son claramente más largos; los de hora pico, más cortos. Los **usuarios frecuentes hacen viajes más cortos** (uso como transporte), los esporádicos viajes más largos (uso recreativo/turístico). Los features creados capturan información que no estaba explícita.

# 13. Discretización (clase 5)
Aplicamos dos métodos vistos en clase:
1. **Ancho fijo por dominio** → `franja_horaria` (madrugada 0–6, mañana 6–12, tarde 12–18, noche 18–24): más interpretable que 24 horas.
2. **Puntos de corte por cuantiles** → `rango_distancia` en 5 grupos con igual cantidad de viajes (`KBinsDiscretizer`, *fit* en train). Adecuado porque la distancia es asimétrica (intervalos de igual ancho dejarían grupos casi vacíos).

In [ ]:
FRANJAS = ['madrugada', 'mañana', 'tarde', 'noche']
for d_ in (train, test):
    d_['franja_horaria'] = pd.cut(d_['hora'], bins=[-1, 5, 11, 17, 23], labels=FRANJAS)

kbins = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile', subsample=None)
train['rango_distancia'] = kbins.fit_transform(train[['distancia_km']]).ravel().astype(int)   # CALCULAR + aplicar
test['rango_distancia'] = kbins.transform(test[['distancia_km']]).ravel().astype(int)         # APLICAR
print('Cortes de distancia (km) calculados en train:', np.round(kbins.bin_edges_[0], 2))

fig, axes = plt.subplots(1, 3, figsize=(19, 4))
train['franja_horaria'].value_counts().reindex(FRANJAS).plot.bar(ax=axes[0], color='steelblue'); axes[0].set_title('Franja horaria (ancho fijo)')
train['rango_distancia'].value_counts().sort_index().plot.bar(ax=axes[1], color='steelblue'); axes[1].set_title('Rango de distancia (cuantiles)')
sns.boxplot(data=train.sample(MUESTRA, random_state=SEED), x='rango_distancia', y='duracion_min', ax=axes[2], showfliers=False)
axes[2].set_title('Duración por rango de distancia')
for ax in axes: ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

Cortes obtenidos en train: 0,93 · 1,47 · 2,1 · 3,13 km. Se ve una relación **monótona** clara: a mayor rango de distancia, mayor duración. El primer grupo incluye los viajes de distancia 0 (vuelven a la misma estación), con duraciones muy dispersas.

# 14. Codificación de variables categóricas (clase 4)
La técnica depende del **tipo de variable** y de su **cardinalidad**:

| Variable | Tipo / cardinalidad | Codificación | Motivo |
|---|---|---|---|
| `modelo_bicicleta` | Binaria | 0/1 (una columna) | Recomendación para binarias |
| `genero` | Nominal, 4 categorías | **One-hot** | Baja cardinalidad, sin orden |
| `franja_horaria` | Nominal, 4 categorías | **One-hot** | Baja cardinalidad |
| `hora`, `dia_semana`, `mes` | Cíclicas | **Cyclic encoding** (seno y coseno) | Las 23 h están "cerca" de las 0 h; diciembre de enero |
| `rango_distancia` | Ordinal | **Ordinal** (0–4, ya numérica) | Conserva el orden |
| `id_estacion_origen` / `destino` | Nominal, ~400 categorías | **Target encoding** (media de duración por estación, *fit* en train, con suavizado) | Alta cardinalidad: one-hot crearía 800 columnas. El target encoding resume cada estación en un número relacionado con la target |

**Cuidado con el target encoding:** tiene riesgo de sobreajuste (sobre todo en categorías con pocos datos). Lo mitigamos con **suavizado**: mezclamos la media de la estación con la media global, ponderando por la cantidad de viajes de la estación.

In [ ]:
# --- Binaria
for d_ in (train, test):
    d_['modelo_iconic'] = (d_['modelo_bicicleta'] == 'ICONIC').astype(int)

# --- One-hot (categorías definidas en TRAIN; si en test aparece una categoría nueva, queda en 0)
def one_hot(d_, col, categorias):
    for c in categorias:
        d_[f'{col}_{c}'] = (d_[col] == c).astype(int)
    return d_
cats_genero = sorted(train['genero'].unique())
for d_ in (train, test):
    one_hot(d_, 'genero', cats_genero)
    one_hot(d_, 'franja_horaria', FRANJAS)

# --- Cíclica
for d_ in (train, test):
    for col, periodo in [('hora', 24), ('dia_semana', 7), ('mes', 12)]:
        d_[f'{col}_sin'] = np.sin(2 * np.pi * d_[col] / periodo)
        d_[f'{col}_cos'] = np.cos(2 * np.pi * d_[col] / periodo)

# --- Target encoding con suavizado (CALCULAR en train, APLICAR en test)
media_global = train['duracion_min'].mean()
def target_encoding(col, k=50):
    g = train.groupby(col)['duracion_min'].agg(['mean', 'count'])
    return (g['count'] * g['mean'] + k * media_global) / (g['count'] + k)
for col in ['id_estacion_origen', 'id_estacion_destino']:
    mapa = target_encoding(col)
    nombre = 'te_' + col.replace('id_', '')
    train[nombre] = train[col].map(mapa)
    test[nombre] = test[col].map(mapa).fillna(media_global)   # estación no vista en train -> media global
    print(f'{col}: estaciones en test que no estaban en train = {(~test[col].isin(mapa.index)).sum()} viajes')

# Visualizamos el encoding cíclico de la hora
h = pd.DataFrame({'hora': range(24)})
plt.figure(figsize=(4.5, 4.5))
plt.scatter(np.sin(2*np.pi*h.hora/24), np.cos(2*np.pi*h.hora/24), c=h.hora, cmap='twilight')
for i in [0, 6, 12, 18, 23]:
    plt.annotate(f'{i} h', (np.sin(2*np.pi*i/24), np.cos(2*np.pi*i/24)))
plt.title('Cyclic encoding de la hora'); plt.xlabel('hora_sin'); plt.ylabel('hora_cos'); plt.axis('equal'); plt.show()

# 15. Escalado (clase 5)
- Usamos **estandarización (z-score)**: media 0 y desvío 1. La elegimos frente a la normalización min-max porque varias features tienen colas largas (`viajes_usuario`, `distancia_km`) y min-max quedaría dominado por los máximos. Además **PCA requiere estandarizar**.
- Escalamos **solo las numéricas continuas**. No escalamos: one-hot y binarias (ya son 0/1), cíclicas (ya están entre −1 y 1; escalarlas rompería el círculo), ordinal `rango_distancia` (rango chico 0–4).
- `viajes_usuario` tiene una cola muy larga → primero le aplicamos **log(1+x)** (transformación para reducir el impacto de extremos) y después estandarizamos.
- El scaler se ajusta (*fit*) en TRAIN y se aplica en TEST. Si en test aparece un valor fuera del rango de train (*data drift*) no es un problema para la estandarización.

In [ ]:
for d_ in (train, test):
    d_['log_viajes_usuario'] = np.log1p(d_['viajes_usuario'])

NUM_ESCALAR = ['distancia_km', 'lat_estacion_origen', 'long_estacion_origen',
               'lat_estacion_destino', 'long_estacion_destino',
               'log_viajes_usuario', 'te_estacion_origen', 'te_estacion_destino']

scaler = StandardScaler()
train_esc = pd.DataFrame(scaler.fit_transform(train[NUM_ESCALAR]), columns=NUM_ESCALAR, index=train.index)  # CALCULAR + APLICAR
test_esc = pd.DataFrame(scaler.transform(test[NUM_ESCALAR]), columns=NUM_ESCALAR, index=test.index)        # APLICAR

comparacion = pd.concat({'antes (train)': train[NUM_ESCALAR].agg(['mean', 'std']).T,
                         'después (train)': train_esc.agg(['mean', 'std']).T,
                         'después (test)': test_esc.agg(['mean', 'std']).T}, axis=1)
display(comparacion.round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
train[NUM_ESCALAR].sample(20_000, random_state=SEED).boxplot(ax=axes[0], rot=60); axes[0].set_title('Antes de estandarizar')
train_esc.sample(20_000, random_state=SEED).boxplot(ax=axes[1], rot=60); axes[1].set_title('Después de estandarizar')
plt.tight_layout(); plt.show()

train[NUM_ESCALAR] = train_esc
test[NUM_ESCALAR] = test_esc

Después de estandarizar todas las variables quedan en la **misma escala** (media 0, desvío 1 en train; en test valores muy parecidos porque se usaron los parámetros de train). La correlación de Pearson entre variables no cambia con el escalado.

# 16. Desbalance de clases: no aplica (problema de regresión)
La consigna pide analizar el desbalance **"en el caso que se trate de un problema de clasificación"**. El nuestro es de **regresión**: la target es continua y no hay clases que puedan estar sub- o sobrerrepresentadas, así que este paso **no corresponde**.

Lo que sí cumple un rol parecido en regresión es la **distribución de la target**: la duración es muy asimétrica (muchos viajes cortos, pocos muy largos). Eso ya lo tratamos con:
- el **capping** en el p99 de train (sección 11), para que los viajes extremos no dominen el modelo;
- la opción de predecir **log(duración)** (sección 8), que vuelve la distribución casi simétrica;
- el **split estratificado por tramos de duración** (sección 9), para que train y test tengan la misma proporción de viajes largos.

*Si el problema fuera de clasificación* (por ejemplo, "¿el viaje supera los 30 minutos?"), mediríamos el desbalance con proporciones, entropía de Shannon e índice de Gini, y lo trataríamos **solo en train** con undersampling (sobran datos) o con pesos por clase, sin balancear nunca el test.

# 17. Selección de features (clase 6)
Seguimos los 4 criterios de clase:
- **(A)** features irrelevantes, no disponibles o no confiables (criterio y dominio).
- **(B)** features **redundantes** (asociación feature–feature).
- **(C)** features con relación **demasiado fuerte** con la target (posible leakage).
- **(D)** **relevancia predictiva** (información mutua, tests F / chi²).

Trabajamos sobre una muestra de train para los cálculos pesados.

### (A) Criterio y conocimiento del dominio
Ya descartamos en la sección 7–8: `fecha_destino` (no disponible al predecir: leakage), `id_recorrido` (irrelevante), nombres y direcciones (redundantes), `id_usuario` crudo. Las coordenadas de las estaciones quedan como candidatas.

### (B) Redundancia entre features

In [ ]:
CANDIDATAS_NUM = ['distancia_km', 'lat_estacion_origen', 'long_estacion_origen', 'lat_estacion_destino',
                  'long_estacion_destino', 'log_viajes_usuario', 'te_estacion_origen', 'te_estacion_destino',
                  'rango_distancia', 'hora', 'dia_semana', 'mes']
CANDIDATAS_BIN = ['mismo_origen_destino', 'es_fin_de_semana', 'es_feriado', 'es_hora_pico', 'modelo_iconic']
CANDIDATAS_CAT = ['genero', 'franja_horaria']

ms = train.sample(MUESTRA, random_state=SEED)
corr = ms[CANDIDATAS_NUM + CANDIDATAS_BIN].corr(method='spearman')
plt.figure(figsize=(13, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1, annot_kws={'size': 8})
plt.title('Correlación de Spearman entre features (muestra de train)'); plt.show()

pares = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
             .rename('spearman').reset_index().assign(abs=lambda x: x.spearman.abs())
             .sort_values('abs', ascending=False).head(8))
pares

In [ ]:
# Redundancia entre categóricas: V de Cramér
cat_cols = CANDIDATAS_CAT + ['dia_semana', 'es_fin_de_semana', 'es_hora_pico', 'rango_distancia', 'mismo_origen_destino']
cv = pd.DataFrame(index=cat_cols, columns=cat_cols, dtype=float)
for a in cat_cols:
    for b in cat_cols:
        cv.loc[a, b] = 1.0 if a == b else cramers_v(ms[a], ms[b])[0]
plt.figure(figsize=(8, 6.5))
sns.heatmap(cv, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1)
plt.title("V de Cramér entre variables categóricas"); plt.show()

**Redundancias encontradas:**
- `franja_horaria` ↔ `hora`: la franja es una discretización de la hora (información repetida) → nos quedamos con la hora en **encoding cíclico** (más detalle) y descartamos la franja para el modelo.
- `dia_semana` ↔ `es_fin_de_semana`: muy asociados → conservamos `es_fin_de_semana` (resume lo esencial que vimos en el EDA) + el encoding cíclico del día.
- `rango_distancia` ↔ `distancia_km`: casi 1 (Spearman) → es la versión discretizada; conservamos la continua.
- `mismo_origen_destino` ↔ `distancia_km` / `rango_distancia`: la distancia 0 ya lo indica, pero lo conservamos porque capta un comportamiento **distinto** (paseo) que un modelo lineal no inferiría de "distancia = 0".
- Coordenadas lat/long de origen y destino: correlación alta entre origen y destino del mismo eje (la gente viaja cerca) → candidatas a resumir con PCA.

### (C) y (D) Asociación con la target y relevancia predictiva

In [ ]:
X_sel = ms[CANDIDATAS_NUM + CANDIDATAS_BIN].copy()
for c in cats_genero: X_sel[f'genero_{c}'] = ms[f'genero_{c}']
y_reg = ms['duracion_min']

# (C) Asociación feature-target (Spearman para regresión)
asoc = X_sel.apply(lambda col: stats.spearmanr(col, y_reg)[0]).rename('spearman_con_target')

# (D) Relevancia predictiva: información mutua y test F (regresión)
ms_mi = ms.sample(50_000, random_state=SEED)   # la información mutua es costosa (usa vecinos cercanos): usamos 50 mil filas
mi = pd.Series(mutual_info_regression(X_sel.loc[ms_mi.index], y_reg.loc[ms_mi.index], random_state=SEED,
                                      discrete_features=[c not in ['distancia_km', 'lat_estacion_origen', 'long_estacion_origen',
                                                                   'lat_estacion_destino', 'long_estacion_destino', 'log_viajes_usuario',
                                                                   'te_estacion_origen', 'te_estacion_destino'] for c in X_sel.columns]),
               index=X_sel.columns, name='info_mutua')
F, pval = f_regression(X_sel, y_reg)
rel = pd.concat([asoc, mi, pd.Series(F, index=X_sel.columns, name='F'), pd.Series(pval, index=X_sel.columns, name='p_valor')], axis=1)
rel['MI_norm'] = rel['info_mutua'] / rel['info_mutua'].max()
rel['F_norm'] = rel['F'] / rel['F'].max()
rel = rel.sort_values('MI_norm', ascending=False)
display(rel.round(4))

rel[['MI_norm', 'F_norm']].plot.barh(figsize=(10, 7)).invert_yaxis()
plt.title('Relevancia predictiva para la DURACIÓN (normalizada)'); plt.xlabel('score normalizado (0-1)'); plt.show()

**Interpretación:**
- **(C)** Ninguna feature tiene una relación "demasiado fuerte" con la target (ninguna correlación cercana a 1) → **no hay indicios de leakage** entre las candidatas (el leakage real, la fecha de destino, ya fue excluido).
- **(D)** Las features más relevantes, por información mutua y por F, son: **`distancia_km`** (por lejos), `rango_distancia`, `log_viajes_usuario`, **`te_estacion_destino`** y `te_estacion_origen`, y luego `es_fin_de_semana`, `dia_semana` y `mismo_origen_destino`. Género, feriado, mes y modelo aportan poco.
- No nos quedamos con un solo método: comparamos MI (capta relaciones **no lineales**) con F (solo lineal). Ejemplo claro: las **coordenadas** tienen F casi 0 (no hay relación *lineal*: ir más al norte no alarga el viaje) pero MI apreciable (la *ubicación* sí importa, de forma no lineal) → las conservamos.

**Criterio de reducción y conjunto final:** descartamos las redundantes (`franja_horaria`, `rango_distancia`, `dia_semana` ordinal) y conservamos las de mayor relevancia más las temporales cíclicas. Mantenemos género y modelo (aportan poco, pero son baratos y podrían interactuar con otras en modelos de árbol); se podrían quitar en una segunda iteración con validación del modelo.

In [ ]:
FEATURES_FINALES = (
    ['distancia_km', 'mismo_origen_destino', 'te_estacion_origen', 'te_estacion_destino',
     'lat_estacion_origen', 'long_estacion_origen', 'lat_estacion_destino', 'long_estacion_destino',
     'log_viajes_usuario', 'es_fin_de_semana', 'es_feriado', 'es_hora_pico', 'modelo_iconic',
     'hora_sin', 'hora_cos', 'dia_semana_sin', 'dia_semana_cos', 'mes_sin', 'mes_cos']
    + [f'genero_{c}' for c in cats_genero])

X_train, y_train = train[FEATURES_FINALES], train['duracion_min']
X_test, y_test = test[FEATURES_FINALES], test['duracion_min']

print('Features finales:', len(FEATURES_FINALES))
print('X_train:', X_train.shape, '| X_test:', X_test.shape)

# 18. Extracción de features: PCA (clase 7)
PCA busca **nuevos ejes** (componentes principales) ordenados por la cantidad de **varianza** que capturan. Pasos:
1. Datos **estandarizados** (ya lo hicimos).
2. Matriz de covarianza → **autovectores** (direcciones / *loadings*) y **autovalores** (varianza explicada).
3. Elegir k componentes: **método del codo** o acumular **70–80% de la varianza**.

**Lo aplicamos a las features numéricas continuas** (coordenadas, distancia, target encodings, frecuencia de usuario). No incluimos binarias / one-hot / cíclicas: PCA asume relaciones lineales y trabaja con varianza, que en variables 0/1 no tiene el mismo significado. *Fit* en TRAIN, *transform* en TEST.

In [ ]:
PCA_COLS = NUM_ESCALAR   # 8 numéricas continuas, ya estandarizadas
pca = PCA(random_state=SEED)
pca.fit(X_train[PCA_COLS])

var = pca.explained_variance_ratio_
var_acum = np.cumsum(var)
k80 = int(np.argmax(var_acum >= 0.80) + 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].bar(range(1, len(var) + 1), var * 100, color='steelblue')
axes[0].plot(range(1, len(var) + 1), var * 100, 'o-', color='firebrick')
axes[0].set_title('Scree plot (método del codo)'); axes[0].set_xlabel('Componente'); axes[0].set_ylabel('% varianza explicada')
axes[1].plot(range(1, len(var) + 1), var_acum * 100, 'o-')
axes[1].axhline(80, color='red', ls='--', label='80%'); axes[1].axvline(k80, color='gray', ls=':')
axes[1].set_title(f'Varianza acumulada: {k80} componentes alcanzan el 80%'); axes[1].set_xlabel('Componentes'); axes[1].legend()
plt.tight_layout(); plt.show()

print('Autovalores:', np.round(pca.explained_variance_, 3))
pd.DataFrame({'% varianza': var * 100, '% acumulado': var_acum * 100}, index=[f'PC{i+1}' for i in range(len(var))]).round(1)

In [ ]:
loadings = pd.DataFrame(pca.components_[:k80].T, index=PCA_COLS, columns=[f'PC{i+1}' for i in range(k80)])
plt.figure(figsize=(1.6 * k80 + 3, 5))
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Loadings: cuánto aporta cada variable a cada componente'); plt.show()

# Scores: datos proyectados en PC1-PC2, coloreados por la duración
pca_k = PCA(n_components=k80, random_state=SEED).fit(X_train[PCA_COLS])
muestra_pca = train.sample(30_000, random_state=SEED)
scores = pca_k.transform(muestra_pca[PCA_COLS])
plt.figure(figsize=(8.5, 6))
sc = plt.scatter(scores[:, 0], scores[:, 1], c=muestra_pca['duracion_min'], cmap='viridis', s=5, alpha=.5,
                 norm=matplotlib.colors.LogNorm())
plt.colorbar(sc, label='Duración (min, escala log)')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('Viajes proyectados en PC1–PC2, color = duración (muestra de train)'); plt.show()

# Aplicamos la reducción a train y test
pcs_train = pd.DataFrame(pca_k.transform(X_train[PCA_COLS]), index=X_train.index, columns=[f'PC{i+1}' for i in range(k80)])
pcs_test = pd.DataFrame(pca_k.transform(X_test[PCA_COLS]), index=X_test.index, columns=[f'PC{i+1}' for i in range(k80)])
X_train_pca = pd.concat([X_train.drop(columns=PCA_COLS), pcs_train], axis=1)
X_test_pca = pd.concat([X_test.drop(columns=PCA_COLS), pcs_test], axis=1)
print(f'Dimensiones: {X_train.shape[1]} features -> {X_train_pca.shape[1]} features con PCA '
      f'({len(PCA_COLS)} numéricas reemplazadas por {k80} componentes, {var_acum[k80-1]:.0%} de su varianza)')

**Interpretación de PCA:**
- Las componentes resumen grupos de variables relacionadas (ver loadings): **PC1** y **PC3** combinan las **4 coordenadas** (origen y destino juntas, porque están muy correlacionadas): representan la **ubicación del viaje en la ciudad** en dos ejes diagonales. **PC2** agrupa los **target encodings** de origen y destino ("estaciones donde los viajes suelen ser largos", en oposición a la frecuencia de uso). **PC4** es casi pura **distancia** y **PC5** casi pura **frecuencia del usuario**.
- Con PC1–PC4 se llega al 77,5%; con **5 componentes se retiene ≈ 88%** de la varianza de las 8 variables numéricas (criterio 70–80% y codo alrededor de PC4–PC5). El último autovalor (PC8 ≈ 1,7%) confirma la redundancia entre coordenadas.
- En el gráfico PC1–PC2 los viajes cortos y largos **se mezclan**: PCA no ordena los datos según la duración porque busca varianza, **no** relación con la target (es no supervisado).

**Ventajas:** menos dimensiones (menor costo, menos riesgo de *curse of dimensionality*); elimina redundancia (las coordenadas origen/destino están correlacionadas); componentes no correlacionadas entre sí (útil para modelos lineales); permite visualizar en 2D.

**Desventajas:** se pierde ~20% de la información; las componentes son **difíciles de interpretar** (¿qué significa "PC2 = 1,3"?), cuando la distancia o la estación son fáciles de explicar; PCA solo capta relaciones **lineales**; no considera la target, por lo que puede descartar varianza que sí era predictiva.

**Conclusión sobre la reducción:** en este problema la dimensionalidad ya es moderada (~23 features) frente a millones de filas, así que la reducción **no es imprescindible**. Dejamos ambas versiones (`X_train` y `X_train_pca`) para comparar en la etapa de modelado; por interpretabilidad preferiríamos la versión con selección de features.

# 19. Conclusiones

**Sobre el dataset**
- 3,56 M de viajes de Ecobici en 2024, 17 variables, mayoría categóricas y fechas. Uso **fuertemente laboral**: 87% en días hábiles, picos a las 8 y 17–18 h, más uso en primavera.
- La duración es **muy asimétrica** (mediana 15 min, cola de días) → usar estadística robusta (mediana, IQR) e IQR para outliers en lugar de 3σ.

**Calidad de datos (desafíos encontrados)**
- 1 duplicado. Faltantes en `fecha_destino` (0,09%) que **no son MCAR** sino **MAR**: aparecen todos desde el 7/11/2024 (falla/cambio del sistema) y sin duración → eliminados por no tener target.
- Faltantes en `genero` (0,34%) de 535 usuarios sin perfil completo, comportamiento ≈ MCAR respecto del viaje → categoría `NO_INFORMADO`.
- Faltantes "disfrazados" y errores: duraciones 0, **devoluciones inmediatas** (~250 mil), velocidades imposibles, viajes > 6 h (bici retenida de noche o no devuelta: fuera del alcance del problema). En total se eliminó ~10% de registros que no son viajes válidos.
- **Leakage detectado**: `fecha_destino` define la target → excluida.

**Preparación aplicada (siempre calculando en train y aplicando en test)**
- Split 80/20 estratificado · nulos con nueva categoría · capping p99 de la target · features nuevos (distancia, temporales, feriados, hora pico, frecuencia de usuario) · discretización (ancho fijo y cuantiles) · codificación (binaria, one-hot, cíclica, ordinal, target encoding) · estandarización · desbalance: no aplica por ser regresión · selección de features (Spearman, V de Cramér, eta², información mutua, F) · PCA.

**Hallazgos para el modelo**
- La **distancia** entre estaciones es la feature más predictiva; luego la estación de destino/origen, si el viaje vuelve al mismo lugar, la frecuencia de uso del usuario y el fin de semana.
- Aun así, la relación es ruidosa: el mismo trayecto puede ser un traslado directo o un paseo. Esperamos que un modelo **no lineal** (ej.: árboles / boosting) funcione mejor que uno lineal, y que predecir `log(duración)` ayude a los modelos lineales.
- PCA reduce dimensiones pero a costa de interpretabilidad; con pocas features y muchos datos no es indispensable.

In [ ]:
# Resumen final de los conjuntos listos para entrenar
pd.DataFrame({
    'filas': [len(X_train), len(X_test), len(X_train_pca), len(X_test_pca)],
    'columnas': [X_train.shape[1], X_test.shape[1], X_train_pca.shape[1], X_test_pca.shape[1]],
}, index=['X_train', 'X_test', 'X_train_pca', 'X_test_pca'])